# E1.6 · Operating vs outcome guardrails

**Function E — AI for GRC → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.5 · Evaluation output as audit evidence](https://spbreed.github.io/cyber-commons/lessons/E1.5.html)**.

| | |
|---|---|
| Open-source tooling | NeMo Guardrails, LLM Guard |
| Open-weight models | Llama Guard 4 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Two different kinds of control get confused constantly. One bounds how the system runs — budgets, scopes, approvals. The other bounds what it produces. They are tested differently and they fail differently.

## 2 · The framework

```
   operating guardrails            outcome guardrails
   +----------------------+        +-----------------------+
   | HOW it runs          |        | WHAT it produces      |
   | budgets, scopes,     |        | content, decisions,   |
   | approvals, sandbox   |        | actions taken         |
   +----------------------+        +-----------------------+
   tested by attempting    tested by sampling outputs
   the forbidden action    against a rubric

   different tests, different failure modes, constantly confused
```

Guardrails come in two kinds, and confusing them is how a programme passes audit
while missing harm.

**Operating guardrails** constrain *how the system runs*: all egress through the
gateway, privileged tools gated below L3, every action logged. They are testable
today, cheap to verify, and produce clean evidence.

**Outcome guardrails** constrain *what results are acceptable*: no unrecoverable
customer data loss, no increase in customer-facing incidents, no disparate
outcomes across segments. They matter more and most need a measurement you do
not yet have.

The failure is not choosing one. It is shipping only the first column, reporting
it as coverage, and never labelling the second column as unmeasured.

## 3 · Demo — classify a real guardrail set

In [ ]:
def classify(rule, constrains_outcome, measurement_exists):
    kind = "outcome" if constrains_outcome else "operating"
    if kind == "operating":
        return {"rule": rule, "kind": kind, "enforceable_today": True,
                "risk": "may satisfy audit while missing real harm"}
    return {"rule": rule, "kind": kind, "enforceable_today": measurement_exists,
            "risk": ("enforceable" if measurement_exists
                     else "needs an agreed measurement before it can be enforced")}

RULES = [
 ("all agent egress goes through the gateway", False, True),
 ("privileged tools require approval below L3", False, True),
 ("every action is logged with the acting identity", False, True),
 ("agent identities are separately revocable", False, True),
 ("no agent action causes unrecoverable customer data loss", True, False),
 ("automated remediation does not increase customer-facing incidents", True, True),
 ("model outputs do not produce disparate outcomes across segments", True, False),
]
print(f"{'rule':60s}{'kind':11s}{'enforceable':>12}")
print("-" * 86)
for rule, outcome, measurable in RULES:
    c = classify(rule, outcome, measurable)
    print(f"{c['rule']:60s}{c['kind']:11s}{str(c['enforceable_today']):>12}")

## 4 · Where it breaks — the coverage number that lies

In [ ]:
operating = [r for r in RULES if not r[1]]
outcome   = [r for r in RULES if r[1]]
enforceable_outcome = [r for r in outcome if r[2]]

print(f"operating guardrails : {len(operating)}  all enforceable today")
print(f"outcome guardrails   : {len(outcome)}  of which enforceable: "
      f"{len(enforceable_outcome)}")

naive = len(operating) / len(RULES)
honest = (len(operating) + len(enforceable_outcome)) / len(RULES)
print(f"\n'guardrail coverage' if you count only what you shipped: "
      f"{len(operating)}/{len(operating)} = 100%")
print(f"coverage across ALL agreed guardrails: "
      f"{len(operating)+len(enforceable_outcome)}/{len(RULES)} = {honest:.0%}")
print("\nThe first number is what usually reaches a steering committee.")

## 5 · The control — define the measurement, or label it unmeasured

In [ ]:
def specify_outcome_guardrail(rule, metric, threshold, source, cadence):
    complete = all([metric, threshold is not None, source, cadence])
    return {"rule": rule, "metric": metric, "threshold": threshold,
            "source": source, "cadence": cadence,
            "status": "enforceable" if complete else "ASPIRATION — label it as such"}

SPECS = [
 specify_outcome_guardrail(
   "automated remediation does not increase customer-facing incidents",
   metric="customer-facing SEV1+SEV2 per 1000 remediations",
   threshold=1.2, source="incident management system", cadence="monthly"),
 specify_outcome_guardrail(
   "no agent action causes unrecoverable customer data loss",
   metric="", threshold=None, source="", cadence=""),
]
for s in SPECS:
    print(f"{s['rule']}")
    print(f"   metric   {s['metric'] or '—'}")
    print(f"   threshold {s['threshold'] if s['threshold'] is not None else '—'}")
    print(f"   source   {s['source'] or '—'}")
    print(f"   status   {s['status']}\n")

def programme_statement(rules, specs):
    enforceable = len([r for r in rules if not r[1]]) + \
                  len([s for s in specs if s["status"] == "enforceable"])
    aspirations = [s["rule"] for s in specs if s["status"] != "enforceable"]
    return (f"{enforceable}/{len(rules)} guardrails are enforceable today.\n"
            f"The following are agreed but UNMEASURED, and are not counted as "
            f"coverage:\n" + "\n".join(f"   - {a}" for a in aspirations))
print(programme_statement(RULES, SPECS))
assert any(s["status"] != "enforceable" for s in SPECS)

## What you just proved

Four operating guardrails are all enforceable today; three outcome guardrails are enforceable only where a measurement exists. Counting only what shipped gives 100% coverage; counting all agreed guardrails gives 71%. One outcome guardrail is fully specified and enforceable; the other is labelled an aspiration and excluded from coverage.

## Your turn

Pick one outcome guardrail your programme has agreed and specify its metric, threshold, source and cadence precisely enough that someone could dispute the result. If you cannot, say so in the coverage report rather than counting it.

---

**Next → [E1.7 · Continuous control verification](https://spbreed.github.io/cyber-commons/lessons/E1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*